# Held-out end-to-end evaluation (CTC-unseen, open-vocabulary)

With the shared speaker partition (`data/splits.json`), `Phone2TextTranscription.ipynb`'s own TEST
split is already CTC-unseen. This notebook is the **stronger cross-corpus check**: it runs the full
pipeline `wav → CTC phones → P2G text` on `AutorskieDane/AutorskiDataset` — literary clips in
different recording conditions from a corpus the CTC **never trained on at all**.

It reports WER/CER (raw and normalized) for the P2G model vs the no-LM heuristic baseline, the upstream
CTC phone-error rate (PER, the acoustic ceiling), and a **stop/go verdict**: ship P2G only if it beats
the heuristic on this unseen audio. Long clips are chunked on silence (not truncated), and targets are
compared under normalization so the contest is fair.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

import torch

from src.ctc.config import CTCConfig as C
from src.ctc.inference import load_ctc_model, wav_to_phone_labels
from src.ctc.textgrid import textgrid_to_phone_ids
from src.ctc.metrics import compute_per
from src.p2g.model import P2GModel
from src.p2g.metrics import corpus_wer, corpus_cer
from src.p2g.data import normalize_text
from src.p2g.transcribe import chunk_phone_labels
from src.p2g.config import P2GConfig as P
from src.wordmaker import phonemes_to_text
from src.utils.device import get_device

# --- Config -----------------------------------------------------------------
ROOT = Path.cwd().parent
# Speaker-disjoint CTC (TrainingCTC.ipynb) — never saw these speakers OR this corpus.
CTC_CHECKPOINT = ROOT / "trained_models" / "ctc_speaker_disjoint.pt"
SAVE_DIR = ROOT / "trained_models" / "p2g_plt5_small"   # fine-tuned P2G model
EVAL_DIR = ROOT / "AutorskieDane" / "AutorskiDataset"   # CTC-unseen held-out audio
NUM_BEAMS = 4
MAX_CHUNK = max(1, P.MAX_SOURCE_LEN - 16)               # phone-chunk size for long clips
DEVICE = get_device()
print("device   :", DEVICE)
print("eval dir :", EVAL_DIR)

## 1. Load the trained CTC and P2G models

The fine-tuned P2G model is produced by `Phone2TextTranscription.ipynb` (section 3). The CTC
checkpoint ships with the repo.

In [ ]:
if not SAVE_DIR.exists():
    raise FileNotFoundError(
        f"Fine-tuned P2G model not found at {SAVE_DIR}.\n"
        "Run Phone2TextTranscription.ipynb (section 3) first to produce it."
    )

ctc_model = load_ctc_model(str(CTC_CHECKPOINT), DEVICE)
p2g = P2GModel.from_pretrained(str(SAVE_DIR), device=DEVICE)
print("loaded CTC + P2G models")

## 2. Transcribe every held-out clip

For each `wav`: run the CTC model → greedy phones, then (a) the P2G seq2seq for the real prediction
and (b) the no-LM heuristic (split on silence + `wordmaker.phonemes_to_text`) for the baseline. The
reference is the sibling `.txt` transcript. Where a `.TextGrid` exists we also keep the phone-id
sequences to measure the upstream CTC PER.

In [ ]:
def heuristic_text(phone_string):
    """Split phones on silence into words, map each to letters, join with spaces."""
    words, cur = [], []
    for ph in phone_string.split():
        if ph in ("sil", "sp"):
            if cur:
                words.append(phonemes_to_text(cur, after_silence=True))
                cur = []
        else:
            cur.append(ph)
    if cur:
        words.append(phonemes_to_text(cur, after_silence=True))
    return " ".join(w for w in words if w)


def p2g_chunked(labels):
    """Transcribe via P2G, chunking long phone streams on silence so they don't truncate."""
    chunks = chunk_phone_labels(labels, MAX_CHUNK)
    texts = [p2g.transcribe(c, num_beams=NUM_BEAMS) for c in chunks]
    return " ".join(t for t in texts if t.strip())


refs, p2g_preds, base_preds = [], [], []
ctc_pred_ids, ctc_tgt_ids = [], []  # only for clips that have a TextGrid

wavs = sorted(EVAL_DIR.glob("*.wav"))
for wav in wavs:
    txt = wav.with_suffix(".txt")
    if not txt.exists():
        continue
    ref = " ".join(txt.read_text(encoding="utf-8").split())

    # One CTC forward pass; reuse the phone labels for P2G, baseline, and PER.
    labels = wav_to_phone_labels(str(wav), ctc_model, DEVICE)
    refs.append(ref)
    p2g_preds.append(p2g_chunked(labels))
    base_preds.append(heuristic_text(" ".join(labels)))

    tg = wav.with_suffix(".TextGrid")
    if tg.exists():
        ctc_pred_ids.append([C.LABEL2IDX[l] for l in labels if l in C.LABEL2IDX])
        ctc_tgt_ids.append(textgrid_to_phone_ids(str(tg), map_sp_to_sil=True))

print(f"evaluated {len(refs)} clips ({len(ctc_tgt_ids)} with a TextGrid for PER)")

## 3. Honest WER / CER

Reported both **raw** (case + punctuation sensitive) and **normalized** (lowercased,
punctuation-stripped via `p2g.data.normalize_text`). Normalized is the fairer figure for an
open-vocabulary transcriber. The CTC PER shows how corrupted the phone stream already is on unseen
audio — the upstream ceiling the text model has to work under.

In [ ]:
def report(name, preds, references):
    print(f"{name:24s} WER {corpus_wer(preds, references):.3f}  CER {corpus_cer(preds, references):.3f}")

print("=== raw (case + punctuation sensitive) ===")
report("P2G (plt5-small)", p2g_preds, refs)
report("heuristic baseline", base_preds, refs)

norm_refs = [normalize_text(r) for r in refs]
norm_p2g = [normalize_text(p) for p in p2g_preds]
norm_base = [normalize_text(p) for p in base_preds]
print("\n=== normalized (lowercased, punctuation-stripped) — the fair figure ===")
report("P2G (plt5-small)", norm_p2g, norm_refs)
report("heuristic baseline", norm_base, norm_refs)

# Keep the normalized WERs for the stop/go gate below.
p2g_wer = corpus_wer(norm_p2g, norm_refs)
base_wer = corpus_wer(norm_base, norm_refs)

if ctc_tgt_ids:
    per = compute_per(ctc_pred_ids, ctc_tgt_ids)
    print(f"\nupstream CTC phone error rate (PER): {per:.3f}  ({len(ctc_tgt_ids)} clips)")

In [ ]:
# Example transcriptions: reference vs heuristic vs P2G.
for ref, base, p in list(zip(refs, base_preds, p2g_preds))[:10]:
    print("REF :", ref[:110])
    print("BASE:", base[:110])
    print("P2G :", p[:110])
    print("-" * 80)

In [ ]:
# --- Stop/go gate -----------------------------------------------------------
# The project goal is to beat the no-LM heuristic on unseen audio. Ship P2G only
# if its normalized WER is actually lower; otherwise the honest call is to ship
# the heuristic (or pursue the contingency: KenLM/lexicon rescoring of phones).
delta = base_wer - p2g_wer
print(f"normalized WER  P2G {p2g_wer:.3f}  vs  heuristic {base_wer:.3f}  (Δ={delta:+.3f})")
if p2g_wer < base_wer:
    print("VERDICT: P2G beats the baseline on CTC-unseen audio -> ship P2G.")
else:
    print("VERDICT: P2G does NOT beat the baseline -> ship the heuristic, or try the")
    print("         contingency (phone-lattice / KenLM rescoring). CTC PER is the ceiling.")

## Caveats

* **Long utterances are now chunked** on silence into `MAX_SOURCE_LEN`-sized pieces (`p2g_chunked`),
  so multi-sentence clips no longer truncate at the source window — matching the sentence-length units
  P2G trains on.
* **Domain shift.** Literary/poetic Polish (Kordian, etc.) is out-of-distribution for `plt5-small`'s
  web-text pretraining, so these numbers are a *lower bound* on what cleaner, in-domain audio gives.
* **CTC PER is the ceiling.** On unseen audio the acoustic stage dominates the error; if the verdict is
  "heuristic wins", the highest-leverage fix is a better CTC (more and more-diverse speakers, stronger
  augmentation), not a bigger text model.
* This is the honest cross-corpus counterpart to the in-corpus TEST split: here the CTC never saw the
  audio *or* the corpus, so the acoustic stage is not flattered.